# 🐍 Chapter 12 — Python UDFs (User Defined Functions) and what they cost

A **UDF** is a function *you* write, handed to Spark so it can be used on a DataFrame column
just like a built-in one. Writing one is easy — and that is the trap. A Python UDF is the one
thing in PySpark that pulls your own Python process into the middle of the data path, so this
chapter is as much a **warning label** as a how-to.

Source: the *UDF* video of the PySpark playlist —
[youtu.be/bbNUiWfAP90](https://www.youtube.com/watch?v=bbNUiWfAP90&list=PL2IsFZBGM_IHCl9zhRVC1EXTomkEp_1zm&index=18)

> ⚠️ **Where to run this chapter — run it in the Docker Jupyter container, on `local[*]`.**
> A UDF needs a working Python on whichever machine executes it, and this repo has three
> environments that are *not* interchangeable:
>
> | Environment | Python | UDFs? |
> |-------------|--------|-------|
> | Local `.venv` on Windows | 3.12 + PySpark 3.3.0 | ❌ nothing runs — cloudpickle cannot serialise a function on 3.12, so even `createDataFrame` on a plain list dies with `PicklingError: Could not serialize object` |
> | Docker Jupyter container, `.master("local[*]")` | 3.7.17 | ✅ everything in this chapter, pandas UDFs included |
> | Docker standalone cluster, `.master("spark://bd-spark-master:7077")` | workers on 3.7.10 | ⚠️ plain UDFs only, and only after the worker fix below — pandas UDFs cannot run there |
>
> (**Pickle** = Python's built-in way of turning an object into a stream of bytes so it can be
> stored or sent elsewhere; **unpickle** turns those bytes back into an object.)
>
> **The two cluster-mode gotchas**, both worth understanding because they are what section 2 looks
> like in real life:
> 1. **The python path is baked into the shipped UDF.** The driver's `PYSPARK_PYTHON`
>    (`/usr/local/bin/python` in the Jupyter image) travels with the function, and the executor
>    tries to launch *that exact path*. The Alpine workers only have `/usr/bin/python3`, so every
>    task dies with `Cannot run program "/usr/local/bin/python": error=2, No such file or
>    directory`. Fix: `docker exec bd-spark-worker-1 ln -sf /usr/bin/python3 /usr/local/bin/python`
>    (and the same for worker-2) — that is what the commented-out `ln -s` line in
>    `docker-images/docker-compose.yml` was for.
> 2. **pandas UDFs need `pandas` + `pyarrow` on *every* executor.** The Jupyter image can install
>    them (`pip install pandas==1.3.5 pyarrow==12.0.1`), but the workers are Alpine 3.10 + musl
>    Python 3.7, which has no binary wheels for either — so on this cluster, keep pandas UDFs to
>    `local[*]`.
>
> The `explain()` cells are safe anywhere: they only print a plan, they never execute one.

| # | Section | Question it answers |
|---|---------|---------------------|
| 1 | What a UDF is | What am I actually creating, and why must it be registered? |
| 2 | The journey of one row | What happens on the worker node when my UDF runs? |
| 3 | The four costs | It works — so why is that not good enough? |
| 4 | Memory & OOM | Why does a UDF blow up a node with `OutOfMemoryError`? |
| 5 | Writing one | The three ways to declare a UDF, and the traps |
| 6 | Fix 1 — built-ins | Higher-order functions instead of a UDF |
| 7 | Fix 2 — Scala/Java UDF | Reuse a JVM UDF from Python — no python worker at all |
| 8 | Fix 3 — pandas UDF | Still Python, but a whole batch at a time |
| 9 | Cheat sheet | What to reach for, in what order |


In [ ]:
from pyspark.sql import SparkSession

# run this in the Docker Jupyter container - local[*] means driver and executor are the SAME
# container, so the python that runs your UDF is the one you already have.
# (spark://bd-spark-master:7077 also works for plain UDFs, but only after symlinking python on
#  the workers - see the warning box above. pandas UDFs will not run there at all.)
spark = (SparkSession.builder
         .appName("UDFs")
         .master("local[*]")
         .getOrCreate())
spark


In [ ]:
# Emp Data & Schema
emp_data = [
    ["001","101","John Doe","30","Male","50000","2015-01-01"],
    ["002","101","Jane Smith","25","Female","45000","2016-02-15"],
    ["003","102","Bob Brown","35","Male","55000","2014-05-01"],
    ["004","102","Alice Lee","28","Female","48000","2017-09-30"],
    ["005","103","Jack Chan","40","Male","60000","2013-04-01"],
    ["006","103","Jill Wong","32","Female","52000","2018-07-01"],
    ["007","101","James Johnson","42","Male","70000","2012-03-15"],
    ["008","102","Kate Kim","29","Female","51000","2019-10-01"],
    ["009","103","Tom Tan","33","Male","58000","2016-06-01"],
    ["010","104","Lisa Lee","27","Female","47000","2018-08-01"]
]
emp_schema = "employee_id string, department_id string, name string, age string, gender string, salary string, hire_date string"

emp = spark.createDataFrame(data=emp_data, schema=emp_schema)

# salary comes in as a string (course convention) - cast it once so the UDFs below get numbers
from pyspark.sql.functions import col
emp = emp.withColumn("salary", col("salary").cast("int"))

emp.show(truncate=False)


## 📝 1. What is a UDF?

**A UDF (User Defined Function) is a Python function you write yourself and then *register* with
Spark, so Spark can call it on every row of a column — in the exact place a built-in function like
`upper()` or `when()` would go.**

Unpacking that:

- **You only need one when nothing built-in does the job.** `pyspark.sql.functions` already has
  hundreds of functions (`when`, `regexp_replace`, `to_date`, `split`, `concat_ws`, …). Those run
  inside Spark's own engine. A UDF is the escape hatch for logic that genuinely has no built-in —
  a company-specific parsing rule, a checksum, a call into a Python library.
- **You must declare the return type.** Spark works out the schema of the result *before* a single
  row is read, and it cannot inspect your Python function to guess. If you omit the type it assumes
  `StringType`; if you declare the wrong one you silently get `null`s, not an error.
- **It runs row by row, in a *separate Python process*, on the executor** — not in the driver where
  you wrote it, and not inside the executor's JVM either. Section 2 is that whole story.
- **It is a black box to the optimiser.** **Catalyst** (Spark's optimiser — the component that
  rewrites your plan into a faster equivalent one) can read `salary > 50000` and push it down into
  the file scan. It cannot read inside `my_udf(salary)`, so all it can do is run it, on every row
  it is given.

#### Why "register"? Why can't I just call my function?

**Because your function works on one value, and a DataFrame column is not a value.**

Your `def` is written for a single number: `bump(50000)` → `55000.0`. One value in, one value out.
But in a DataFrame you never hold `50000` in your hand — you hold `col("salary")`, which is not a
number at all. It is a **Column object: a *description*, "the salary column of this DataFrame"**.
Hand that to your plain function and Python tries `float(Column) * 1.10` and raises an error,
because a description is not something you can multiply.

So `udf()` does two things a plain `def` cannot:

- **It changes the shape of the function.** `bump` takes a number and returns a number.
  `bump_udf` takes a **Column** and returns a **Column** — an expression Spark can drop into its
  query plan, in the very slot where `upper(col("name"))` would sit.
- **It packages the function for travel.** Your function is *pickled* so it can be shipped to every
  executor (section 2). A `def` sitting in your notebook means nothing to a JVM on another machine.

#### And what does "called on every row" mean?

It means **Spark writes the loop, not you** — out on the executors, where the data already is:

```text
 you write:
     emp.select(bump_udf(col("salary")))

 Spark runs, on each executor, over its own partition:
     for each row in the partition:
         value  = row.salary       # a real Python number now, e.g. 50000
         result = bump(value)      # YOUR function, called once for this row
         put result into the new column for that row
```

10 rows → your function is called 10 times. 10 million rows → 10 million separate Python calls.
You never write that loop and never see it, which is exactly why the cost in section 3 stays
invisible until it hurts.

> Why not just loop yourself — `for row in emp.collect(): bump(row.salary)`? Because `collect()`
> drags the whole dataset into the driver and processes it on one machine. That throws away every
> bit of Spark: the parallelism, the partitions, the cluster.


In [ ]:
# 1) a first UDF - give everyone a 10% raise
'''-- SQL cannot express arbitrary Python logic; the closest built-in version of this rule is:
   SELECT name, salary * 1.10 AS new_salary FROM emp'''

from pyspark.sql.functions import udf, col
from pyspark.sql.types import DoubleType

def bump(salary):                      # a plain Python function - Spark knows nothing about it yet
    if salary is None:                 # SQL NULL arrives as Python None - always guard for it
        return None
    return float(salary) * 1.10

bump_udf = udf(bump, DoubleType())     # now it is a Column expression Spark can put in the plan

emp.select("name", "salary", bump_udf(col("salary")).alias("new_salary")).show(5)


In [ ]:
# 1b) WHY the wrapper is needed - watch the plain function fail on a Column
print("bump(50000) =", bump(50000))      # fine: a plain number goes in, a number comes out

print("col('salary') is a", type(col("salary")))   # not a number - a description of a column

try:
    emp.select(bump(col("salary"))).show(3)        # the PLAIN function, not the UDF
except Exception as e:
    print("plain function on a Column ->", type(e).__name__, ":", e)

# bump_udf is the same logic wrapped so it accepts a Column and returns a Column:
print("bump_udf(col('salary')) is a", type(bump_udf(col("salary"))))


## 📝 2. The journey of one row — what really happens when a UDF runs

Nothing about a UDF happens where you wrote it. Follow one row through the whole trip:

1. **You call the UDF inside a transformation** — `emp.select(bump_udf("salary"))`. Nothing runs;
   Spark only records it in the plan (this is **lazy evaluation**: transformations build the plan,
   an action such as `show()` / `count()` / `write` triggers it).
2. **On the action, the driver ships your logic to the workers.** Spark *pickles* your function
   together with its **closure** (any variable from outside the function that the function uses) and
   sends those bytes along with every task. This is what "Spark copies the logic to the worker nodes"
   means — the **code travels to the data**, never the other way round.
3. **The executor starts a python worker process.** An executor is a **JVM** (Java Virtual Machine —
   the process that runs Java/Scala bytecode). Your Python cannot run *inside* a JVM, so the executor
   launches a **separate operating-system process** on the same node (`python -m pyspark.daemon`),
   one per concurrently running task slot. It is reused by later tasks, but the first one pays the
   startup cost.
4. **Rows are serialised out of the JVM.** **Serialise** = turn objects in memory into a flat stream
   of bytes so another process can read them. JVM row objects → bytes → local socket → the python
   worker **deserialises** them back into Python objects.
5. **Your function runs one row at a time.** One Python call per row. No batching, no vectorising,
   and none of the compiled JVM code Spark normally generates for built-ins.
6. **Results are serialised back.** Python objects → bytes → socket → the JVM deserialises them into
   rows, and the rest of the plan (filters, joins, aggregation, writing) continues inside the JVM.
7. **The JVM reports the finished task to the driver**, exactly as it would for any other task. The
   driver never talks to the python worker.

### The parts involved

Two Python processes exist in a PySpark job that uses a UDF: the one you type in (the driver), and
one *per executor* out on the cluster. They are different processes running the same function.

```text
 ┌───────────────────────────────────┐
 │ DRIVER PROGRAM                    │
 │  ┌────────────┐   ┌────────────┐  │
 │  │ JVM        │←──│ Python     │  │
 │  │ plan +     │   │ your code  │  │
 │  │ scheduling │   │ (notebook) │  │
 │  └────────────┘   └────────────┘  │
 └─────────────────┬─────────────────┘
                   │ tasks + the pickled UDF
 ┌─────────────────▼─────────────────┐
 │ CLUSTER                           │
 │  ┌─────────────────────────────┐  │
 │  │ NODE 1                      │  │
 │  │  ┌──────────┐ ┌──────────┐  │  │
 │  │  │ JVM      │ │ python   │  │  │
 │  │  │ executor │ │ worker   │  │  │
 │  │  └──────────┘ └──────────┘  │  │
 │  └─────────────────────────────┘  │
 │  ┌─────────────────────────────┐  │
 │  │ NODE 2                      │  │
 │  │  ┌──────────┐ ┌──────────┐  │  │
 │  │  │ JVM      │ │ python   │  │  │
 │  │  │ executor │ │ worker   │  │  │
 │  │  └──────────┘ └──────────┘  │  │
 │  └─────────────────────────────┘  │
 └───────────────────────────────────┘
```

The python worker sits **beside** the executor JVM on the same node — inside the same box in the
picture, but **not inside the JVM**. That one detail is the source of every problem in sections 3
and 4.

### The same thing as a sequence, inside one executor

```text
 EXECUTOR JVM   (one task = one partition)
 ┌────────────────────────────────────────────┐
 │ 1. read the partition, rows live in the    │
 │    JVM heap                                │
 │ 2. serialise a batch of rows (pickle)      │
 └───────────────────┬────────────────────────┘
                     │ local socket
 ┌───────────────────▼────────────────────────┐
 │ PYTHON WORKER   (separate OS process)      │
 │ 3. start up, unpickle your function        │
 │ 4. run YOUR code - ONE ROW AT A TIME       │
 │ 5. serialise the results                   │
 └───────────────────┬────────────────────────┘
                     │ local socket
 ┌───────────────────▼────────────────────────┐
 │ EXECUTOR JVM                               │
 │ 6. deserialise results, finish the plan    │
 │ 7. report the task result to the DRIVER    │
 └────────────────────────────────────────────┘
```


In [ ]:
# 2) proof in the query plan: BatchEvalPython = the round trip out to a python worker
'''SELECT UPPER(name) FROM emp        -- a built-in: never leaves the JVM'''
from pyspark.sql.functions import upper

print(">>> built-in upper() - all JVM, no python worker")
emp.select(upper("name")).explain()

print(">>> python UDF - look for 'BatchEvalPython'")
emp.select(bump_udf(col("salary"))).explain()

# BatchEvalPython [bump(salary)] is the step where rows leave the JVM, cross into the python
# worker and come back. Spotting it in a plan is the standard way to find a slow UDF.


## 📝 3. The four costs of a Python UDF

Every one of these comes straight out of the seven steps above.

#### Cost 1 — serialisation, twice, for every row

Data that was already sitting in the executor's memory has to be converted to bytes, pushed through
a socket, rebuilt as Python objects, then converted back the other way. A built-in function does
none of this: the rows never leave the JVM.

#### Cost 2 — an extra process per executor

The executor must launch and feed a python worker. It costs startup time, and it costs memory that
your `--executor-memory` setting never accounted for (section 4).

#### Cost 3 — one row at a time

Spark's built-ins are compiled into JVM code that runs over whole columns. A Python UDF is a Python
function call per row — interpreted, one at a time. On millions of rows the difference is not a few
percent, it is a different order of magnitude.

#### Cost 4 — the optimiser goes blind

Catalyst treats the UDF as an opaque box, which costs you more than speed:

- **No pushdown.** `emp.filter(my_udf(col("x")) == "y")` cannot be pushed into the file scan or the
  database, so Spark reads everything and filters afterwards. `col("x") == "y"` would have been
  pushed down.
- **No reordering, no folding.** Spark cannot decide the UDF is cheap enough to run early or
  expensive enough to run last — it has no idea what is inside.
- **It can run more often than you think.** Spark may evaluate a UDF before the condition you
  assumed protects it, so `when(col("x").isNotNull(), my_udf("x"))` does **not** guarantee your
  function never sees a `None`. Guard inside the function.


## 📝 4. Why a UDF can take the whole node down with OutOfMemory

**Spark sizes and tracks the memory of the executor *JVM* — the python worker is a different
process, so the memory your Python code uses is outside everything Spark measures.**

- **`--executor-memory 4g` sets the JVM heap only.** *Heap* = the memory region a JVM manages and
  garbage-collects. Spark knows how much of it is used, spills to disk when it fills, and reports it
  in the UI.
- **The python worker's memory is not in that budget.** It is an ordinary OS process taking RAM from
  whatever the node has left. On YARN or Kubernetes that space is `spark.executor.memoryOverhead`;
  in standalone mode it is simply the machine's free memory.
- **Spark cannot limit it by default.** The JVM cannot garbage-collect another process' memory. The
  one dial that exists is `spark.executor.pyspark.memory` — **unset by default**, i.e. uncapped as
  far as Spark is concerned. (The video's "Spark has no control over the Python process" is the
  right idea; precisely, Spark can start it, stop it and — if you set that config — cap it, but it
  can never manage what is inside it.)
- **What makes it grow:** loading a big lookup dictionary, a model, or a pandas frame inside the
  function; accumulating state in a global; a library that caches. Multiply by the number of task
  slots on the node — every concurrent task has its own python worker.
- **The failure doesn't look like a Spark error.** The OS (or the container runtime) kills the
  process, and you see `Python worker exited unexpectedly (crashed)`, a lost executor, or a
  container `Killed by YARN for exceeding memory limits` — with no useful Python traceback.

```text
 ONE WORKER NODE  (say 8 GB of RAM)
 ┌────────────────────────────────────────────────┐
 │ executor JVM                                   │
 │  ┌──────────────────────────────────────────┐  │
 │  │ heap = --executor-memory 4g              │  │
 │  │ Spark tracks and manages every byte here │  │
 │  └──────────────────────────────────────────┘  │
 │                                                │
 │ python workers   (OUTSIDE the heap)            │
 │  ┌──────────────────────────────────────────┐  │
 │  │ your dict / model / pandas frame / cache │  │
 │  │ not measured, not capped by default      │  │
 │  └──────────────────────────────────────────┘  │
 └────────────────────────────────────────────────┘
   heap 4g  +  python growing with no cap
   ---> the node runs out of RAM
   ---> the OS kills the process ---> the task fails
```


## 📝 5. Writing a UDF — the three ways, and the traps

Same function, three registrations. Which one you pick only changes *where you can call it from*.

- **`udf(fn, ReturnType())`** — returns a Column expression for the DataFrame API. Section 1 used it.
- **`@udf(returnType=...)` decorator** — identical, written on top of the `def`. The name is now the
  UDF; the plain Python function is no longer directly callable.
- **`spark.udf.register("name", fn, ReturnType())`** — puts it in the **SQL function registry**, so
  it can be used in `spark.sql("SELECT name(...)")` and in `expr("name(...)")`. It *also* returns a
  Column-style UDF, so one call gives you both worlds.

Traps worth knowing before you write your first real one:

- **Return type must match reality.** Return a Python `str` from a UDF declared `IntegerType()` and
  you get `null` in every row — no exception, no warning.
- **`None` will arrive.** SQL `NULL` becomes Python `None`. An unguarded `s.upper()` raises, and one
  raised exception fails the task, and after `spark.task.maxFailures` retries, the whole job.
- **Everything the function touches gets pickled.** A large object referenced from the enclosing
  scope is shipped with *every task*; use `spark.sparkContext.broadcast(obj)` to send one read-only
  copy per executor instead. A `SparkSession` or DataFrame referenced inside a UDF cannot be pickled
  at all — a UDF runs on the executor, where there is no session.
- **Spark assumes a UDF is deterministic** (same input → same output) and may reorder or re-run it.
  If it isn't (random, time, a network call), mark it `udf(...).asNondeterministic()`.
- **The executors need the same Python you have — at the same path, with the same libraries.** The
  driver's `PYSPARK_PYTHON` path is *baked into the shipped function*, and the executor launches
  exactly that path; a missing binary gives `Cannot run program ".../python": error=2`, and a
  missing library gives a `ModuleNotFoundError` from a machine you never touched. Driver and
  executor Python must also agree on the major.minor version. Nothing about a UDF is checked until
  it runs on the cluster — this chapter's warning box is a live example of both failures.


In [ ]:
# 3) the decorator form
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

@udf(returnType=StringType())
def salary_band(salary):
    if salary is None:
        return "unknown"
    elif salary >= 65000:
        return "senior"
    elif salary >= 50000:
        return "mid"
    else:
        return "junior"

emp.select("name", "salary", salary_band(col("salary")).alias("band")).show(5)


In [ ]:
# 4) registering the SAME function for SQL
'''SELECT name, salary, salary_band_sql(salary) AS band FROM emp'''
from pyspark.sql.functions import expr

def band(salary):
    if salary is None:
        return "unknown"
    elif salary >= 65000:
        return "senior"
    elif salary >= 50000:
        return "mid"
    else:
        return "junior"

spark.udf.register("salary_band_sql", band, StringType())   # now it exists in the SQL registry

emp.createOrReplaceTempView("emp")
spark.sql("SELECT name, salary, salary_band_sql(salary) AS band FROM emp").show(5)

# ... and the same registered name works inside expr() on the DataFrame API
emp.select("name", expr("salary_band_sql(salary) AS band")).show(5)


## 📝 6. Fix 1 (the best one) — use built-ins, including *higher-order* functions

**The first answer to "should I write a UDF?" is almost always "no — a built-in already does this".**
Built-ins are executed by Spark's own engine inside the JVM: no python worker, no serialisation, and
Catalyst can still optimise around them.

`salary_band` above is a good example of a UDF that never needed to exist — `when()` / `otherwise()`
expresses exactly the same branching, and the plan stays inside the JVM.

#### What "higher-order function" means

**A higher-order function is a function that takes another function as one of its arguments.** In
Spark that's the array functions — `transform`, `filter`, `exists`, `aggregate` — where you pass a
small lambda: `transform(nums, x -> x * 2)`.

The important part: **that lambda is a Spark SQL expression, not Python code.** Even when you write
it as a Python `lambda` in PySpark, you are building a Column expression that Spark compiles into
the plan — it is never pickled and never shipped to a python worker. So "loop over the elements of
an array and change each one", the classic reason people reach for a UDF, is a built-in operation
that costs nothing extra.


In [ ]:
# 5) the built-in replacement for the salary_band UDF - no python worker at all
'''SELECT name, salary,
          CASE WHEN salary >= 65000 THEN 'senior'
               WHEN salary >= 50000 THEN 'mid'
               ELSE 'junior' END AS band
   FROM emp'''
from pyspark.sql.functions import when

band_builtin = (when(col("salary") >= 65000, "senior")
                .when(col("salary") >= 50000, "mid")
                .otherwise("junior"))

emp.select("name", "salary", band_builtin.alias("band")).show(5)

# compare the two plans: the UDF version has BatchEvalPython, this one does not
print(">>> UDF version")
emp.select(salary_band(col("salary"))).explain()
print(">>> built-in version")
emp.select(band_builtin).explain()


In [ ]:
# 6) higher-order functions on an array column - the lambda runs in the JVM
'''SELECT nums,
          transform(nums, x -> x * 2)              AS doubled,
          filter(nums, x -> x > 2)                 AS big,
          aggregate(nums, 0, (acc, x) -> acc + x)  AS total
   FROM nums_tbl'''
from pyspark.sql import functions as F

nums = spark.createDataFrame([(1, [1, 2, 3, 4]), (2, [5, 6])], "id int, nums array<int>")

# written as SQL strings ...
nums.select(
    "nums",
    F.expr("transform(nums, x -> x * 2)").alias("doubled"),
    F.expr("filter(nums, x -> x > 2)").alias("big"),
    F.expr("aggregate(nums, 0, (acc, x) -> acc + x)").alias("total"),
).show(truncate=False)

# ... or with PySpark's wrappers, where the Python lambda BUILDS a Column expression
# (it is evaluated once, at plan time - it is not shipped to a python worker)
nums.select(
    "nums",
    F.transform("nums", lambda x: x * 2).alias("doubled"),
    F.filter("nums", lambda x: x > 2).alias("big"),
    F.aggregate("nums", F.lit(0), lambda acc, x: acc + x).alias("total"),
).show(truncate=False)


## 📝 7. Fix 2 — write the UDF in Scala/Java and call it from PySpark

**If the logic really has no built-in, write it in the JVM's own language: a Scala or Java UDF is
registered *inside* the executor JVM, so calling it from PySpark starts no python worker and
serialises nothing.** Your Python code just names it.

Why this works: the barrier is not "Python the language", it is the **process boundary**. A Scala
UDF lives in the same process as the data, so it is an ordinary function call — same cost class as
a built-in (it is still opaque to Catalyst, so Cost 4 from section 3 still applies).

**Step 1 — write it in Scala**

```scala
package com.example.udfs

import org.apache.spark.sql.api.java.UDF1

class SalaryBand extends UDF1[Integer, String] {
  override def call(salary: Integer): String = {
    if (salary == null) "unknown"
    else if (salary >= 65000) "senior"
    else if (salary >= 50000) "mid"
    else "junior"
  }
}
```

**Step 2 — build a jar** (`sbt package` / `mvn package`). A **jar** is a zip of compiled JVM
classes — the unit of code you hand to Spark.

**Step 3 — give the jar to the session**, so the class exists in the driver *and* every executor:

```bash
spark-submit --jars /path/udfs.jar my_job.py
# or, from a notebook, before the session is created:
#   SparkSession.builder.config("spark.jars", "/path/udfs.jar")
```

**Step 4 — register the class from Python and use it by name** — see the next cell.


In [ ]:
# 7) calling a Scala/Java UDF from PySpark  (template - needs the jar from step 2/3)
'''SELECT name, salary, salary_band_scala(salary) AS band FROM emp'''

# spark.udf.registerJavaFunction(
#     "salary_band_scala",          # the SQL name you will call it by
#     "com.example.udfs.SalaryBand",  # fully-qualified class name inside the jar
#     StringType(),                 # return type (optional; Spark can infer it from UDF1[..])
# )
#
# spark.sql("SELECT name, salary, salary_band_scala(salary) AS band FROM emp").show(5)
# emp.select("name", expr("salary_band_scala(salary) AS band")).show(5)
#
# The plan for this has NO BatchEvalPython step - the work happens inside the executor JVM.
# Note the asymmetry: from Python you can only call it BY NAME through the SQL registry
# (registerJavaFunction), which is why the SQL-string style shows up in every example of this.


## 📝 8. Fix 3 — pandas UDFs, when it has to be Python

**A pandas UDF (also called a vectorised UDF) is still your Python function on the executor, but
Spark hands it a whole *batch* of rows as a pandas `Series` instead of calling it once per row.**

- **It fixes Cost 1 and Cost 3, not the others.** The transfer uses **Apache Arrow** — a columnar
  in-memory format that both the JVM and Python understand, so a batch is moved with almost no
  conversion work instead of pickling row by row. And your code runs once per batch, letting pandas
  and NumPy do the loop in compiled code.
- **The python worker is still there**, still outside the JVM heap — section 4's memory story applies
  unchanged, and in fact a whole batch now sits in Python memory at once
  (`spark.sql.execution.arrow.maxRecordsPerBatch`, default 10000, is the dial).
- **Catalyst is still blind to it** — Cost 4 stands.
- **Requirements:** `pyarrow` and `pandas` installed on the driver *and* every executor. Type hints
  are how Spark 3.x knows which flavour you mean: `pd.Series -> pd.Series` is a scalar pandas UDF.
- **Rule of the return value:** the Series you return must be the same length as the one you got.

> 🐳 **In this repo:** the Docker Jupyter image ships neither library, so install them there once —
> `docker exec bd-pyspark-jupyter-lab pip install pandas==1.3.5 pyarrow==12.0.1` (those are the last
> versions supporting its Python 3.7). That covers `local[*]`, where the driver *is* the executor.
> It does **not** extend to `spark://` runs: the Alpine/musl workers have no wheels available, so a
> pandas UDF there fails with a `PythonException` from the executor. And the install is lost the
> moment the container is recreated — a two-line Dockerfile makes it stick.

So the ladder is: built-in → higher-order function → Scala/Java UDF → pandas UDF → plain Python UDF.


In [ ]:
# 8) the same 10% raise as a vectorised pandas UDF
'''SELECT name, salary * 1.10 AS new_salary FROM emp'''
import pandas as pd
from pyspark.sql.functions import pandas_udf

@pandas_udf("double")                              # return type as a DDL string
def bump_vec(salary: pd.Series) -> pd.Series:      # a WHOLE COLUMN of a batch, not one value
    return salary * 1.10                           # vectorised: pandas loops in compiled code

emp.select("name", "salary", bump_vec(col("salary")).alias("new_salary")).show(5)

# in the plan this shows as ArrowEvalPython (instead of BatchEvalPython) - still a trip out
# to the python worker, but an Arrow-shaped, batch-at-a-time one
emp.select(bump_vec(col("salary"))).explain()


## 📝 9. Cheat sheet

**The demerits of a plain Python UDF, in one list:** rows are serialised out and deserialised back;
an extra python process must be started and fed; your code runs one row at a time; Catalyst cannot
optimise around it; and its memory sits outside the JVM heap where Spark can neither see nor cap it.

| Reach for | When | Cost |
|-----------|------|------|
| **Built-in function** (`when`, `regexp_replace`, `to_date`, `split`, …) | Almost always — check the docs first | Runs inside the JVM, fully optimisable |
| **Higher-order function** (`transform`, `filter`, `exists`, `aggregate`) | Per-element work on an `array<...>` column | Same as a built-in — the lambda is a plan expression, not Python |
| **Scala/Java UDF** + `registerJavaFunction` | Logic with no built-in, on a hot path, and you can build a jar | JVM-native call; opaque to Catalyst but nothing else |
| **pandas UDF** (`@pandas_udf`) | It must be Python (a NumPy/pandas/ML library call) | Arrow + batch-at-a-time; python worker still outside the heap |
| **Python UDF** (`@udf`) | Small data, prototyping, or logic that genuinely cannot be vectorised | Everything in the list above |

**Handy checks**

- `df.explain()` → `BatchEvalPython` = a Python UDF is in the plan; `ArrowEvalPython` = a pandas UDF;
  neither = it all stays in the JVM.
- `spark.catalog.listFunctions()` → what is registered in the SQL registry (yours included).
- `spark.executor.pyspark.memory` → the only Spark-side cap on python worker memory; unset = uncapped.


In [ ]:
# Stop Spark Session
spark.stop()
